In [ ]:
import numpy
import polars
from data_index.iceberg_config import S3TablesCatalogConfig, IcebergTableConfig
from data_index.analysis.tables import IMOS_DATA_LIVE_TABLE
from data_index.analysis.datasets import (
    DATASET,
    DATASET_FILTER,
    get_dataset_objects_df,
    get_dataset_xarray_dataset,
)

In [ ]:
table = IMOS_DATA_LIVE_TABLE.load()

In [ ]:
table.schema

In [ ]:
df = table.scan(
    selected_fields=("bucket", "key", "size", "facility", "last_modified_date",),
    row_filter=(
        "facility == 'SRS'"
    ),
).to_polars()

In [ ]:
dataset: DATASET = "station_lucinda_jetty_daily_satlantic_hyperocr"

In [ ]:
dataset_df = (
    get_dataset_objects_df(
        df=df,
        dataset=dataset,
    )
)
dataset_df

In [ ]:
ds = get_dataset_xarray_dataset(
    dataset=dataset,
    # bucket="imos-optimised-nonproduction",
    # skip_signature=False,
)

In [ ]:
nc_filenames = polars.DataFrame(data={"filename": dataset_df["key"].str.split("/").list.last().unique().sort()})
zarr_filenames = polars.DataFrame(data={"filename": numpy.unique(ds.filename.values)})

In [ ]:
nc_filenames

In [ ]:
zarr_filenames

In [ ]:
set(nc_filenames["filename"]) - set(zarr_filenames["filename"])

In [ ]:
ds.info()